In [1]:
#!/usr/bin/env python3
import torch
from torchdiffeq import odeint

torch.manual_seed(0)
device = "cpu"
dtype = torch.float64

# Simple linear ODE: dy/dt = A y  (analytic solution y(t)=exp(tA) y(0))
A = torch.tensor([[-0.3, 0.2],
                  [-0.1, -0.4]], dtype=dtype, device=device)

def f(t, y):
    return y @ A.T  # shape (..., 2)

# Initial state and time grids
y0 = torch.randn(5, 2, dtype=dtype, device=device)  # batch of 5 vectors
t_fwd = torch.tensor([0.0, 1.0], dtype=dtype, device=device)   # forward: 0 -> 1
t_rev = torch.flip(t_fwd, dims=[0])                             # backward: 1 -> 0

# Forward integrate to t=1
y1 = odeint(f, y0, t_fwd, method="rk4", options={"step_size": 1/200})[-1]

# Backward integrate by FLIPPING THE TIME GRID (no sign flip in f)
y0_back = odeint(f, y1, t_rev, method="rk4", options={"step_size": 1/200})[-1]

# Check error
err = (y0_back - y0).norm(dim=1).max().item()
print(f"max‖y0_back - y0‖ = {err:.3e}  (should be ~1e-8 to 1e-10 with rk4)")

# Optional: show the alternative equivalent way (keep time increasing, negate RHS)
def f_neg(t, y):  # dy/dt = -f(t,y)
    return -f(t, y)

y0_back_alt = odeint(f_neg, y1, t_fwd, method="rk4", options={"step_size": 1/200})[-1]
err_alt = (y0_back_alt - y0).norm(dim=1).max().item()
print(f"max‖y0_back_alt - y0‖ = {err_alt:.3e}  (equivalent alternative)")


max‖y0_back - y0‖ = 8.006e-16  (should be ~1e-8 to 1e-10 with rk4)
max‖y0_back_alt - y0‖ = 4.965e-16  (equivalent alternative)
